# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiezugwu/Data-Science/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
I used Correlation / signal analysis because Spearman correlations of each feature against the label, plus the three mini-tests with verdicts (tier medians, with client halves as a consistency check). Random forest: The one real predictive model: 200 trees predicting severity and Grouped validation- Splits grouped by client, used for both the random forest and the tiny linear model

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import files
uploaded = files.upload()


Saving ranking_lifecycle.parquet to ranking_lifecycle.parquet


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor

D = pd.read_parquet("decline_recovery.parquet")
R = pd.read_parquet("ranking_lifecycle.parquet")[["content_hash_id", "search_volume", "main_intent"]]
D = D.merge(R, on="content_hash_id", how="left")

# label (positive = more severe)
D["severity"] = -(D.impressions_last_30d / D.impressions_prev_30d - 1) * 100

# honest features: earlier windows and static facts only (no product flags, no outcome-window columns)
D["impressions_first30"] = D.impressions_90d - D.impressions_last_30d - D.impressions_prev_30d
D["early_momentum_pct"] = (D.impressions_prev_30d / D.impressions_first30.where(D.impressions_first30 >= 50) - 1) * 100
D["prev30_ctr_pct"] = D.clicks_prev_30d / D.impressions_prev_30d * 100
D["log_prev30_impressions"] = np.log1p(D.impressions_prev_30d)
D["age_at_outcome_start_days"] = D.content_age_days - 30
D["intent"] = D.main_intent.astype("category").cat.codes.replace(-1, np.nan)
D["type"] = D.content_type.astype("category").cat.codes
feats = ["impressions_first30", "early_momentum_pct", "prev30_ctr_pct", "log_prev30_impressions",
         "age_at_outcome_start_days", "search_volume", "intent", "type",
         "keyword_token_count", "public_url_path_depth", "content_title_token_count"]

K = 500   # the one metric: precision@500

def precision_at_k(order_index, is_severe, k=K):
    """share of the first k rows (in the given ranked order) that are truly severe"""
    return is_severe.loc[order_index[:k]].mean()

def run(seed):
    # 1. split rows into train and test, grouped by client
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(gss.split(D, groups=D.client_hash_id))
    train, test = D.iloc[train_idx], D.iloc[test_idx]

    # "severe" = worst quartile, cutoff taken from TRAIN only
    cut = train.severity.quantile(0.75)
    severe = test.severity >= cut

    # 2. train the model on the train set only
    model = RandomForestRegressor(n_estimators=200, min_samples_leaf=50, n_jobs=-1, random_state=42)
    model.fit(train[feats], train.severity)

    # 3. rank the SAME test rows with the frozen baseline and with the model
    b = test.assign(new_page=(test.age_at_outcome_start_days < 60).astype(int),
                    no_history=(test.impressions_first30 < 50).astype(int))
    b["score"] = b.new_page + b.no_history
    base_order = b.sort_values(["score", "impressions_first30", "impressions_prev_30d"],
                               ascending=[False, True, False]).index
    pred = pd.Series(model.predict(test[feats]), index=test.index)
    model_order = pred.sort_values(ascending=False).index

    # 4. same metric, same test rows
    return {"seed": seed, "test_rows": len(test), "test_clients": test.client_hash_id.nunique(),
            "base_rate": severe.mean(),
            "baseline": precision_at_k(base_order, severe),
            "model": precision_at_k(model_order, severe)}

print(pd.Series(run(42)).round(3))

# how stable is it? repeat with other client splits
rows = pd.DataFrame([run(s) for s in (1, 2, 3, 4, 42)]).round(3)
print(rows.to_string(index=False))

seed              42.000
test_rows       8034.000
test_clients      10.000
base_rate          0.299
baseline           0.308
model              0.692
dtype: float64
 seed  test_rows  test_clients  base_rate  baseline  model
    1       2316            10      0.428     0.550  0.598
    2       9390            10      0.308     0.382  0.722
    3       6046            10      0.391     0.324  0.624
    4      10432            10      0.195     0.380  0.486
   42       8034            10      0.299     0.308  0.692


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
Where the model is wrong

The model squeezes everything toward the middle.
Pages that lost 30-50% were predicted at about 56%.
Pages that lost 70-99% were predicted at only 59-64%.
So it gets the order roughly right but not the size of the loss.
About a quarter of its top 500 are total collapses (132 pages that fell to 100% loss), and it catches these well.
Without those pages, precision@500 drops from 0.69 to 0.58. The base rate is 0.29, so the lead is real but smaller.
It is weakest on low-traffic pages. Small numbers make the loss percentage noisy.
It fails on one client, 8ddc46, a batch of 951 new pages. Inside that client it does no better than random.
Of its top 500, 155 were false alarms. Their median actual loss was 52%

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.